# JiT-S2-VMamba — B/C RoPE, 2D axial (row / column), A tied in pairs

Rotary position encoding on the **scan's key and query**, the Mamba analogue of where RoPE sits in attention. (The earlier `feature_rope` rotated the mixer's features, with no counter-rotation, so nothing cancels there.)

The scan computes $y_i=\sum_{j\le i} C_i^\top\big(\prod_{k=j+1}^{i} e^{\Delta_k A}\big) B_j\,\Delta_j x_j$: **$B$ is the key and $C$ the query**. Rotating both in adjacent `d_state` pairs by their cell's 2-D position,

$$(R(\varphi_i)C_i)^\top\,\mathrm{decay}\,R(\varphi_j)B_j \;=\; C_i^\top\,\mathrm{decay}\,R(\varphi_j-\varphi_i)\,B_j,$$

holds **only if the decay is the same for both states of every rotated pair**, i.e. $A$ tied within each pair. Mamba-1 has a free $A[d,n]$ per state, so `a_tie: pairs` stores one $A$ per pair (`A_logs` $(K, d_\text{inner}, N/2)$, init $A=-[1,1,3,3,\dots,15,15]$: the baseline's range), which removes 147,456 parameters.

**Rotation.** Pairs $j<N/4$ turn by the cell's row, the rest by its column:

$$\omega_j=\theta^{-4j/N},\quad j=0,\dots,N/4-1,\qquad a^{\mathrm{row}}_{r,j}=r\,\omega_j,\qquad a^{\mathrm{col}}_{c,j}=c\,\omega_j,$$

feature_rope's 2D axial construction taken over the $N$ = 16 state channels. With only 4 frequencies per axis, $\theta$ = **10** (not 10000) so that all of them turn appreciably across the 8×8 grid: $\omega$ = 1, 0.56, 0.32, 0.18 rad per cell.

**Where it goes.** `… → x_proj → split (Δ_r, B, C) → [ssm_norm] → **rotate B and C** → scan (A tied) → CrossMerge → …`. The tables are permuted with `cross_scan`, so a cell keeps its rotation in all four paths and $C_i^\top B_j$ sees the true 2-D offset between the cells, even where they are one step apart in the scan but far apart on the grid (a row end and the next row start). Δ, B and C still come from unrotated features; the stock kernel is unchanged. The exactness is for the B/C product only: the model still sees absolute position (sin-cos embedding, DWConv padding, scan order).

**Related work.** TransXSSM applies fixed RoPE to B and C in Mamba-2, whose scalar A per head needs no tying; Mamba-3's complex-valued state is the data-dependent version of a state that rotates each step. The 2-D axial placement on SS2D's four paths and the A tying for Mamba-1 are ours.

| model (Tiny-ImageNet, 100 epochs) | A | B / C | params | FID-10K |
|---|---|---|---|---|
| VMamba, d_state 16 (JiM baseline) | free per (channel, state) | unrotated | 31,031,616 | 91.73 |
| control: A tied in pairs (`jit-s2-vmamba-tinyin-atie.ipynb`) | tied in pairs | unrotated | 30,884,160 | ? |
| **this:** B/C RoPE, 2D axial (`jit-s2-vmamba-tinyin-bc-rope-2d-axial.ipynb`) | tied in pairs | rotated by row / column | **30,884,160** | ? |

The rotation adds no parameters, so this run and its control have the same weights at seed 42; the rotation is the only variable between them. Expect a neutral to slightly positive result on 8×8 (the model already gets position from sin-cos, the DWConv and four scan orders); a gap of a point or less needs a second seed of both runs to mean anything.

---

JiT-S2-VMamba on Tiny-ImageNet-200 (64px, patch 8, 200 classes) via the repo's `run_experiment.py` / `evaluate.py`. Comment out the download cell if you attach a dataset instead, and the eval cell if you only want to train.

## 1. Environment  *(Internet ON)*

In [ ]:
import os, sys

# Pick the stack by the image's Python. torch 2.5.1 has no matching torchvision
# for Python 3.13 (Kaggle's newer image), so 3.13 moves to torch 2.6.0; 3.12
# keeps torch 2.5.1 exactly as every earlier run. mamba-ssm 2.2.4 and
# causal-conv1d 1.5.0.post8 publish cp312 and cp313 wheels for both.
PY = f"cp{sys.version_info.major}{sys.version_info.minor}"
if sys.version_info >= (3, 13):
    TORCH, TVISION, TAUDIO, TTAG = "2.6.0", "0.21.0", "2.6.0", "torch2.6"
else:
    TORCH, TVISION, TAUDIO, TTAG = "2.5.1", "0.20.1", "2.5.1", "torch2.5"
print("python", sys.version.split()[0], "->", PY, "| torch", TORCH)

# 1) Pin torch
!pip install -q torch=={TORCH} torchvision=={TVISION} torchaudio=={TAUDIO} \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download wheels with explicit destination
CAUSAL = f"causal_conv1d-1.5.0.post8+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"
MAMBA  = f"mamba_ssm-2.2.4+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"

os.system(f"wget -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/{CAUSAL} -O /kaggle/working/{CAUSAL}")
os.system(f"wget -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/{MAMBA} -O /kaggle/working/{MAMBA}")
for w in (CAUSAL, MAMBA):
    p = f"/kaggle/working/{w}"
    assert os.path.exists(p) and os.path.getsize(p) > 0, f"wheel download failed: {w}"

!pip install -q /kaggle/working/{CAUSAL}
!pip install -q /kaggle/working/{MAMBA}

# 3) Patch mamba-ssm
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"\u2705 Patched {path}")

print(">>> RESTART RUNTIME NOW <<<")

## 2. Repo  *(clone + cd)*

In [ ]:
# Clone the repo. a_tie / bc_rope (SS2D in src/models/vmamba.py), the build_model
# wiring in BOTH run_experiment.py and evaluate.py, the config, the CPU test and the
# GPU check must all be on main.
import os
REPO_DIR = "/kaggle/working/thesis_Choustoulakis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Rodamanthosch/thesis_Choustoulakis.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# sanity: confirm the test, this config and both entry points are present
!test -f tests/test_bc_rope_cpu.py && test -f configs/tiny_imagenet/jit-s2-vmamba-bc-rope-2d-axial.yaml \
    && grep -q 'bc_rope=m.get' scripts/run_experiment.py && grep -q 'a_tie=m.get' scripts/evaluate.py \
    && echo "wiring OK (test + config + train + eval)" || echo "WIRING MISSING -- push the bc_rope commit to main"

## 3. Verify a_tie / bc_rope  *(first session only; ~3-5 min)*

In [ ]:
# === Verify a_tie / bc_rope (CPU proofs, ~1-2 min; first session only -- comment out on resume).
# Reference scan, no GPU: both flags off is bit-identical to the commit before the
# feature (pinned, so this is a real comparison on this clone too) for every arm;
# the tables == an independent float64 derivation; the mixer == a naive float64
# transcription with A tied and B, C rotated per cell; shifting every coordinate
# leaves the B/C RoPE output unchanged with A tied (relative) and moves it with A
# free; only A_logs changes (-147,456 params); train / eval / profile builders.
# PYTHONPATH=. is REQUIRED: without it "from src.models.vmamba import" fails
# and the tests' try/except misreports it as a missing mamba_ssm (known issue).
# If it FAILS, STOP -- do not train a model that is not the one reported.
!PYTHONPATH=. python tests/test_bc_rope_cpu.py

In [ ]:
# === GPU check (~2-3 min; first session only). scripts/check_bc_rope_cuda.py:
# 1) the tables on CUDA vs float64; 2) the mixer (A tied, with and without the
# rotation) on the stock CUDA scan vs mamba_ssm's selective_scan_ref; 3) on the real
# kernel a coordinate shift leaves the B/C RoPE output unchanged (A tied); 4) this
# config fwd+bwd under bf16 at batch 128, finite; 5) it/s and peak memory vs the
# baseline -- note them for the complexity table. If anything FAILS, STOP.
!PYTHONPATH=. python scripts/check_bc_rope_cuda.py --config bc-rope

## 4. Tiny-ImageNet  *(attached dataset, else download to /tmp)*

In [ ]:
# Tiny-ImageNet-200 -> /tmp (EPHEMERAL), ImageFolder layout for train/ and val/.
# Sources, in order: an attached Kaggle dataset containing tiny-imagenet-200/,
# then Stanford (https, http), then the SJTU mirror of the identical zip.
import os, zipfile, glob, shutil, subprocess, hashlib
DATA_DIR = "/tmp/tiny-imagenet-200"
ZIP      = "/tmp/tiny-imagenet-200.zip"
MD5      = "90528d7ca1a48142e341f4ef8d21d0de"
SOURCES  = ["https://cs231n.stanford.edu/tiny-imagenet-200.zip",
            "http://cs231n.stanford.edu/tiny-imagenet-200.zip",
            "https://scidata.sjtu.edu.cn/records/swpcp-01e87/files/tiny-imagenet-200.zip?download=1"]

def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    attached = [os.path.dirname(p) for p in
                glob.glob("/kaggle/input/**/tiny-imagenet-200/train", recursive=True)]
    if attached:
        print("copying attached dataset:", attached[0])
        shutil.copytree(attached[0], DATA_DIR)
    else:
        if os.path.exists(ZIP) and (not zipfile.is_zipfile(ZIP) or _md5(ZIP) != MD5):
            print(f"removing broken download ({os.path.getsize(ZIP)} bytes)")
            os.remove(ZIP)
        for src in SOURCES:
            if os.path.exists(ZIP):
                break
            print("downloading", src, "(~248MB)...")
            r = subprocess.run(["wget", "-q", "--tries=3", "--timeout=60", "-O", ZIP, src])
            ok = r.returncode == 0 and zipfile.is_zipfile(ZIP) and _md5(ZIP) == MD5
            if not ok:
                size = os.path.getsize(ZIP) if os.path.exists(ZIP) else 0
                print(f"  failed (wget exit {r.returncode}, {size} bytes)")
                if os.path.exists(ZIP):
                    os.remove(ZIP)
        assert os.path.exists(ZIP), ("Tiny-ImageNet download failed from every source -- "
                                     "attach a Kaggle tiny-imagenet dataset (Add Input) and rerun this cell")
        print("extracting...")
        with zipfile.ZipFile(ZIP) as z:
            z.extractall("/tmp")

# TRAIN: train/<cls>/images/*.JPEG -> train/<cls>/*.JPEG  (ImageFolder layout)
tr = os.path.join(DATA_DIR, "train")
for cls in os.listdir(tr):
    sub = os.path.join(tr, cls, "images")
    if os.path.isdir(sub):
        for f in glob.glob(os.path.join(sub, "*.JPEG")):
            shutil.move(f, os.path.join(tr, cls))
        shutil.rmtree(sub)
    for b in glob.glob(os.path.join(tr, cls, "*_boxes.txt")):
        os.remove(b)

# VAL: flat val/images + val_annotations.txt -> val/<cls>/*.JPEG  (evaluate.py reads val/)
val = os.path.join(DATA_DIR, "val")
ann = os.path.join(val, "val_annotations.txt")
if os.path.exists(ann):
    with open(ann) as f:
        for line in f:
            img, cls = line.split("\t")[:2]
            os.makedirs(os.path.join(val, cls), exist_ok=True)
            s = os.path.join(val, "images", img)
            if os.path.exists(s):
                shutil.move(s, os.path.join(val, cls, img))
    shutil.rmtree(os.path.join(val, "images"), ignore_errors=True)
    os.remove(ann)

n_tr, n_val = len(glob.glob(tr + "/*")), len(glob.glob(val + "/*"))
print("train classes:", n_tr, "| val classes:", n_val, "| DATA_DIR:", DATA_DIR)
assert n_tr == 200 and n_val == 200, "unexpected class count -- dataset incomplete"

## 5. Settings  *(edits the cloned config; finds the checkpoint to resume)*

In [ ]:
# ── Settings for THIS notebook ─────────────────────────────────────────
A_TIE      = "pairs"      # required by bc_rope: one A per rotated state pair
BC_ROPE    = "2d_axial"   # rotate B and C: first half of the pairs by row, second by column
THETA      = 10.0         # bc_rope_theta: 4 frequencies per axis -> 1, .56, .32, .18 rad/cell
EPOCHS     = 100          # full target; resume across sessions until reached
SAVE_FREQ  = 1            # overwrite checkpoint-last.pt EVERY epoch (12h safety)

# Fallback if the download cell was commented out (attached-dataset workflow):
try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/tiny-imagenet/tiny-imagenet-200"  # <-- your attached dataset
    print("DATA_DIR not set by a download cell; using:", DATA_DIR)

CONFIG  = "configs/tiny_imagenet/jit-s2-vmamba-bc-rope-2d-axial.yaml"
RUN     = "exp_vmamba_bc_rope_2d_axial"
TAG     = "bc-rope"   # must appear in a resumed checkpoint's path (see below)
OUT_DIR = f"/kaggle/working/{RUN}"   # persists in the saved Version

# Sync the cloned config so BOTH training and evaluation read identical values
# (evaluate.py has no model CLI overrides -- it builds the model straight from
# config). adaLN-Zero only, no in-context tokens; compare against
# the a_tie control (jit-s2-vmamba-atie.yaml).
import re
def set_cfg(path, kv):
    s = open(path).read()
    for k, v in kv.items():
        s, n = re.subn(rf"(?m)^(\s*{k}:).*$", rf"\1 {v}", s, count=1)
        assert n == 1, f"key {k!r} not found in {path}"
    open(path, "w").write(s)
set_cfg(CONFIG, {"data_dir": DATA_DIR, "a_tie": A_TIE, "bc_rope": BC_ROPE, "bc_rope_theta": THETA,
                 "d_state": 16, "expand": 1, "ffn": "swiglu"})

# ── Resume across sessions AND accounts (automatic) ────────────────────
# Looks for THIS model's newest checkpoint in: this session's OUT_DIR, any
# attached notebook output (.../checkpoint-last.pt), and any attached DATASET
# holding an uploaded checkpoint (Kaggle unpacks an uploaded .pt into a folder
# with data.pkl / data/ / byteorder / version -- it is zipped back here).
# A checkpoint is used only if its weights match this config's model exactly
# (same names and shapes) AND its path contains TAG ("_" counts as "-"). Shapes
# alone are not enough here: the rotation adds no parameters, so the a_tie control
# and the B/C RoPE run have the SAME state_dict and would load silently. Keep TAG in the name
# of any dataset you upload a checkpoint to.
# The newest epoch wins. To force one, set RESUME_CKPT_FORCE to its path.
RESUME_CKPT_FORCE = None
import os, glob, zipfile, torch, yaml
from scripts.evaluate import build_model
_ref = {k: tuple(v.shape) for k, v in build_model(yaml.safe_load(open(CONFIG))).state_dict().items()}

def _rebuild(folder):
    """Zip an unpacked checkpoint folder back into a loadable .pt in /tmp."""
    dst = os.path.join("/tmp/resume", folder.strip("/").replace("/", "__") + ".pt")
    if not os.path.exists(dst):
        os.makedirs("/tmp/resume", exist_ok=True)
        with zipfile.ZipFile(dst, "w", zipfile.ZIP_STORED) as z:
            for root, _, files in os.walk(folder):
                for f in files:
                    full = os.path.join(root, f)
                    z.write(full, os.path.join("checkpoint", os.path.relpath(full, folder)))
    return dst

cands = [os.path.join(OUT_DIR, "checkpoint-last.pt")]
cands += sorted(glob.glob("/kaggle/input/**/checkpoint-last.pt", recursive=True))
cands += [_rebuild(os.path.dirname(p))
          for p in sorted(glob.glob("/kaggle/input/**/data.pkl", recursive=True))]
best = None
print("checkpoints found:")
for c in ([RESUME_CKPT_FORCE] if RESUME_CKPT_FORCE else cands):
    if not (c and os.path.exists(c)):
        continue
    try:
        p = torch.load(c, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"  unreadable ({type(e).__name__}): {c}")
        continue
    shapes = {k: tuple(v.shape) for k, v in p.get("model", {}).items()} == _ref
    match = shapes and (c == RESUME_CKPT_FORCE or TAG in c.lower().replace("_", "-"))
    label = "THIS model " if match else ("other arm? " if shapes else "other model")
    print(f"  {label}  epoch {p.get('epoch')}  {c}")
    if match and (best is None or p["epoch"] > best[0]):
        best = (p["epoch"], c)
    del p
RESUME_CKPT = best[1] if best else None

print("config :", CONFIG, "|", "bc_rope:", BC_ROPE, "| theta:", THETA, "|", "vmamba d_state 16, adaLN-Zero only")
print("data   :", DATA_DIR, "(exists:", os.path.isdir(DATA_DIR), ")")
print("out    :", OUT_DIR)
print("resume :", f"{RESUME_CKPT} (last finished epoch {best[0]})" if best else "(fresh start)")

## 6. Train  *(Save Version before 12h; next session it resumes by itself)*

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────
# data_dir + the model knobs now live in the (edited) config; we only override run-specific
# knobs here. Runs in a subprocess (fresh torch, no restart).
CMD = (f"python scripts/run_experiment.py --config {CONFIG} "
       f"checkpoint.output_dir={OUT_DIR} checkpoint.save_last_freq={SAVE_FREQ} "
       f"training.epochs={EPOCHS}")
if RESUME_CKPT:
    CMD += f" checkpoint.resume_from={RESUME_CKPT}"
print(CMD, "\n" + "="*70)
!{CMD}

## 7. Evaluate  *(comment out the whole cell to skip)*

In [ ]:
import glob, os

# Find the checkpoint to evaluate: this session's OUT_DIR first, then any
# attached notebook output (Save Version -> attach it as an input). Searches
# instead of hardcoding a notebook hash, so nothing needs editing after the
# first Save Version.
cands = []
for root in (os.path.dirname(OUT_DIR), "/kaggle/input"):
    for which in ("checkpoint-best.pt", "checkpoint-last.pt"):
        cands += sorted(glob.glob(f"{root}/**/{RUN}/{which}", recursive=True))
assert cands, f"no checkpoint found for {RUN} -- train first, or attach the saved Version"
EVAL_CKPT = cands[0]
EVAL_OUT  = f"/kaggle/working/{RUN}/eval"
print("eval ckpt:", EVAL_CKPT)

EVAL_CMD = (f"python scripts/evaluate.py --config {CONFIG} --checkpoint {EVAL_CKPT} "
            f"--n_samples 10000 --batch_size 200 --ema 1 "
            f"--cfg_scale 2.5 --cfg_interval 0.1 1.0 --out_dir {EVAL_OUT}")
print(EVAL_CMD, "\n" + "="*70)
!PYTHONPATH=. {EVAL_CMD}

## Notes
- Checkpoints (`-last`/`-best`/`-ep###`) with both EMA copies + optimizer are in `OUT_DIR` under `/kaggle/working`, so a resumed run equals a continuous one.
- **Resuming, also on another Kaggle account:** attach the previous session's notebook output, or upload its `checkpoint-last.pt` as a dataset **with `bc-rope` in the dataset name** and attach that. The settings cell keeps only checkpoints with this model's shapes and that tag in the path, and resumes from the newest epoch. A checkpoint of the a_tie control has the same shapes and shows as `other arm?`; a baseline checkpoint has a different `A_logs` shape and shows as `other model`. To use a checkpoint regardless, set `RESUME_CKPT_FORCE`. Check the `resume :` line before training.
- **Data:** the Tiny-ImageNet cell uses an attached dataset first, then Stanford, then the SJTU mirror of the identical zip (MD5-checked). To skip downloads for good, upload `tiny-imagenet-200.zip` once as your own Kaggle dataset and attach it.
- Keep `EPOCHS`, `cfg_scale`, `DATA_DIR`, seed and sampler identical to the base run and every other run.
- +0 params vs the control; the cost is one rotation of B and C, (B, 4, 16, 64) each, per block, so the it/s should match the control. Record the it/s and peak memory printed by the GPU check for the complexity table.
- **Comparisons:** this vs `jit-s2-vmamba-tinyin-atie.ipynb` = what the rotation does (same A); this vs the base VMamba run (91.73) = rotation + tying together.
- `model.bc_rope` is `none` | `2d_axial` (needs `a_tie: pairs`, `d_state` divisible by 4, and no in-context tokens, state-init or SSC). Other `bc_rope_theta` values: change `THETA` and `RUN` together, keeping `bc-rope` in the path of a resumed checkpoint.